# Leaflet cluster map of talk locations

This notebook extracts talk locations from `_talks/*.md`, geocodes them using OpenStreetMap (Nominatim), and generates a cluster map using `getorg`.

In [1]:
# Install dependencies
!pip install python-frontmatter getorg geopy --quiet

import frontmatter
import glob
import getorg
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [2]:
# Collect markdown files
g = glob.glob("_talks/*.md")
print("Found files:", len(g))

Found files: 4


In [3]:
# Configuration
TIMEOUT = 20

# Geocoder (IMPORTANT: user_agent required)
geocoder = Nominatim(user_agent="github-actions-talkmap", timeout=TIMEOUT)

location_dict = {}
location = ""
title = ""

## Geocoding talk locations

In [4]:
# Process each talk file
for file in g:
    data = frontmatter.load(file).to_dict()

    if 'location' not in data:
        continue

    title = data.get('title', '').strip()
    venue = data.get('venue', '').strip()
    location = data.get('location', '').strip()

    description = f"{title}<br />{venue}; {location}"

    try:
        result = geocoder.geocode(location)
        if result:
            location_dict[description] = result
            print("OK:", description)
        else:
            print("FAILED:", location)

    except GeocoderTimedOut:
        print("TIMEOUT:", location)
    except Exception as e:
        print("ERROR:", location, e)

OK: Conference Proceeding talk 3 on Relevant Topic in Your Field<br />Testing Institute of America 2014 Annual Conference; Los Angeles, CA, USA


OK: Talk 1 on Relevant Topic in Your Field<br />UC San Francisco, Department of Testing; San Francisco, CA, USA


OK: Talk 2 on Relevant Topic in Your Field<br />London School of Testing; London, UK


OK: Tutorial 1 on Relevant Topic in Your Field<br />UC-Berkeley Institute for Testing Science; Berkeley, CA, USA


In [5]:
# Generate map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(
    location_dict,
    folder_name="talkmap",
    hashed_usernames=False
)

print("Map generated in /talkmap")

Map generated in /talkmap
